In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import os
import re

In [ ]:


# Adjust these paths based on your Drive structure
CSV_PATH = "/content/Annotated_data-Final_data.csv"
IMG_FOLDER_PATH = "/content/drive/MyDrive/LGBTQ Memes/unique_images"
SAVE_PATH = "/content/drive/MyDrive/LGBTQ Memesprocessed_data.csv"

# Load CSV
df = pd.read_csv(CSV_PATH)




In [ ]:
df.head()

In [ ]:
df.columns = df.columns.str.strip()



In [ ]:
df = df[(df['Annotator_1'] != -1) & (df['Annotator_2'] != -1)]
df = df[df['Annotator_1'] == df['Annotator_2']]


In [ ]:
# Drop rows with missing OCR text
df = df.dropna(subset=['Extracted Text'])

# Add final 'label' column
df['label'] = df['Annotator_1'].astype(int)

# Show results
print(df.head())
print(f"[INFO] Cleaned dataset size: {len(df)} rows")

In [ ]:
!pip install textblob


In [ ]:
from textblob import TextBlob

# Function to correct spelling mistakes
def correct_spelling(text):
    if pd.isnull(text):
        return ""
    return str(TextBlob(text).correct())

# Apply to the 'Extracted Text' column
df['corrected_text'] = df['Extracted Text'].apply(correct_spelling)

# Check the result
print(df[['Extracted Text', 'corrected_text']].head())

# we also have autocorrect and pyspellchecker


In [ ]:
!pip install pyspellchecker


In [ ]:
from spellchecker import SpellChecker

# Initialize the spell checker
spell = SpellChecker()

# Function to correct spelling mistakes and handle None or empty strings
def correct_spelling_pyspellchecker(text):
    if not text:
        return ""  # Return an empty string if the text is None or empty
    words = text.split()
    corrected_words = [spell.correction(word) for word in words]
    # Ensure no None values are returned in the list before joining
    corrected_words = [word if word is not None else "" for word in corrected_words]
    return " ".join(corrected_words)

# Drop rows where 'Extracted Text' is NaN
df = df.dropna(subset=['Extracted Text'])

# Apply to the 'Extracted Text' column
df['corrected_text'] = df['Extracted Text'].apply(correct_spelling_pyspellchecker)

# Check the result
print(df[['Extracted Text', 'corrected_text']].head())


In [ ]:
!pip install symspellpy


In [ ]:
!wget https://raw.githubusercontent.com/mammothb/symspellpy/master/symspellpy/frequency_dictionary_en_82_765.txt


In [ ]:
sym_spell.load_dictionary("/content/frequency_dictionary_en_82_765.txt", term_index=0, count_index=1)


In [ ]:
from symspellpy import SymSpell, Verbosity
import pandas as pd
import re
from tqdm import tqdm

# Initialize SymSpell
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

cache = {}
def correct_spelling_symspell(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r'[^\w\s]', '', text)
    corrected_words = []
    for word in text.split():
        if word in cache:
            corrected = cache[word]
        else:
            suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
            corrected = suggestions[0].term if suggestions else word
            cache[word] = corrected
        corrected_words.append(corrected)
    return " ".join(corrected_words)

# Example: Drop NaNs and apply with progress bar
df = df.dropna(subset=['Extracted Text'])
tqdm.pandas()
df['corrected_text'] = df['Extracted Text'].progress_apply(correct_spelling_symspell)

# Preview result
print(df[['Extracted Text', 'corrected_text']].head())


In [ ]:
df.shape

In [ ]:
!pip install Levenshtein

In [ ]:
import numpy as np
import Levenshtein

# Function to calculate Levenshtein Distance
def calculate_levenshtein(text1, text2):
    return Levenshtein.distance(text1, text2)

from tqdm import tqdm
tqdm.pandas()

mask = df['Extracted Text'] != df['corrected_text']
df.loc[mask, 'levenshtein_distance'] = df[mask].progress_apply(
    lambda row: Levenshtein.distance(row['Extracted Text'], row['corrected_text']), axis=1)
df['levenshtein_distance'] = df['levenshtein_distance'].fillna(0).astype(int)

# Print the results
print(df[['Extracted Text', 'corrected_text', 'levenshtein_distance']].head())


In [ ]:
average_levenshtein = df['levenshtein_distance'].mean()
print(f"Average Levenshtein Distance: {average_levenshtein:.2f}")


In [ ]:
# Calculate normalized Levenshtein distance
df['normalized_levenshtein'] = df.apply(
    lambda row: row['levenshtein_distance'] / max(len(row['Extracted Text']), len(row['corrected_text']))
    if max(len(row['Extracted Text']), len(row['corrected_text'])) > 0 else 0,
    axis=1
)

# Compute average normalized distance
average_normalized = df['normalized_levenshtein'].mean()
print(f"Average Normalized Levenshtein Distance: {average_normalized:.4f}")


In [ ]:
df.shape

In [ ]:
# Full image path
df['image_path'] = df['Filename'].apply(lambda x: os.path.join(IMG_FOLDER_PATH, x))


In [ ]:
# Clean OCR text
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text.strip()

df['clean_text'] = df['Extracted Text'].apply(clean_text)


In [ ]:
# Drop rows with missing or unresolved fields
df.dropna(subset=['label', 'clean_text'], inplace=True)


In [ ]:
# Optional: Check if image files actually exist
df = df[df['image_path'].apply(os.path.exists)]


In [ ]:
# Convert label to integer
df['label'] = df['label'].astype(int)


In [ ]:
# Save processed CSV
df.to_csv('/content/Memesprocessed_data.csv', index=False)
print(f"[INFO] Preprocessed data saved at: {'/content/Memesprocessed_data.csv'}")


In [ ]:
import pandas as pd
df = pd.read_csv('/content/Memesprocessed_data.csv')

In [ ]:
df.head()

In [ ]:
# Check the distribution of the 'label' column
label_counts = df['label'].value_counts()

# Calculate the percentage distribution of each label
label_percentage = df['label'].value_counts(normalize=True) * 100

print("Label Counts:")
print(label_counts)

print("\nLabel Percentage Distribution:")
print(label_percentage)


In [ ]:
pip install transformers clip-by-openai


In [ ]:
pip install torch==2.7.0 torchvision==0.22.0


In [ ]:
df.shape

In [ ]:
!pip install git+https://github.com/openai/CLIP.git


In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
IMG_FOLDER_PATH = "/content/drive/MyDrive/LGBTQ Memes/unique_images"

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import torch
import clip
from PIL import Image
import pandas as pd
from tqdm import tqdm

# Load the CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device)

# Function to process and compute embeddings for text
# def get_text_embedding(text):
#     # Encode the text using CLIP
#     text_input = clip.tokenize([text]).to(device)
#     with torch.no_grad():
#         text_features = model.encode_text(text_input)
#     text_features /= text_features.norm(dim=-1, keepdim=True)  # Normalize
#     return text_features.cpu().numpy()
def get_text_embedding(text):
    # Split text into chunks of 77 tokens or less
    max_tokens = 77
    chunks = [text[i:i+max_tokens] for i in range(0, len(text), max_tokens)]

    # Encode each chunk separately and average the embeddings
    embeddings = []
    for chunk in chunks:
        text_input = clip.tokenize([chunk]).to(device)
        with torch.no_grad():
            chunk_features = model.encode_text(text_input)
        chunk_features /= chunk_features.norm(dim=-1, keepdim=True)  # Normalize
        embeddings.append(chunk_features.cpu().numpy())

    # Average the embeddings of all chunks
    embeddings = np.mean(embeddings, axis=0)
    return embeddings

# Function to process and compute embeddings for images
def get_image_embedding(image_path):
    # Open and preprocess the image
    image = Image.open(image_path)
    image_input = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image_input)
    image_features /= image_features.norm(dim=-1, keepdim=True)  # Normalize
    return image_features.cpu().numpy()

# Apply embeddings extraction for each row
def generate_embeddings(df):
    text_embeddings = []
    image_embeddings = []

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        text_embedding = get_text_embedding(row['clean_text'])
        image_embedding = get_image_embedding(row['image_path'])

        text_embeddings.append(text_embedding)
        image_embeddings.append(image_embedding)

    # Add embeddings to the DataFrame
    df['text_embedding'] = text_embeddings
    df['image_embedding'] = image_embeddings
    return df

# # Assuming 'df' is your dataframe loaded from the CSV
# df = generate_embeddings(df)

# # Save the dataframe with embeddings
# df.to_csv('/path_to_save/processed_with_embeddings.csv', index=False)
# print(f"[INFO] Preprocessed data with embeddings saved at: /path_to_save/processed_with_embeddings.csv")


In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into train, validation, and test sets (e.g., 70% train, 15% validation, 15% test)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Now, generate embeddings for each of the splits
train_df = generate_embeddings(train_df)
valid_df = generate_embeddings(valid_df)
test_df = generate_embeddings(test_df)

# Optionally, save each dataframe with embeddings
train_df.to_csv('train_with_embeddings.csv', index=False)
valid_df.to_csv('valid_with_embeddings.csv', index=False)
test_df.to_csv('test_with_embeddings.csv', index=False)


In [ ]:
import pandas as pd

# Load the CSV file
file_path = '/content/train_with_embeddings.csv'
df = pd.read_csv(file_path)

# Show the first few rows of the dataset
print("First 5 rows of the dataset:")
print(df.head())

# Check the basic information (e.g., data types, non-null counts)
print("\nBasic information about the dataset:")
df_info = df.info()

# Display summary statistics of numerical columns
print("\nSummary statistics of numerical columns:")
df_summary = df.describe()

# Check for missing values
print("\nMissing values in the dataset:")
missing_values = df.isnull().sum()

# Check for class distribution in the label column
print("\nClass distribution in the 'label' column:")
class_distribution = df['label'].value_counts()

# Check for unique values in each column (to inspect categorical columns)
print("\nUnique values in each column:")
unique_values = df.nunique()

# Check the distribution of text embeddings and image embeddings (if needed)
# You can inspect the length of the embeddings if they are in text form
print("\nLength of text embeddings in the dataset:")
df['text_embedding_length'] = df['text_embedding'].apply(lambda x: len(eval(x)))  # Assuming embeddings are stored as strings of lists
text_embedding_lengths = df['text_embedding_length'].describe()

print("\nText embedding length summary:")
print(text_embedding_lengths)

# Check for image embeddings (if available)
print("\nLength of image embeddings in the dataset:")
df['image_embedding_length'] = df['image_embedding'].apply(lambda x: len(eval(x)))  # Assuming embeddings are stored as strings of lists
image_embedding_lengths = df['image_embedding_length'].describe()

print("\nImage embedding length summary:")
print(image_embedding_lengths)


In [ ]:
import ast
import numpy as np


In [ ]:
import re
import ast
import numpy as np
import pandas as pd

def clean_embedding_string(embedding_str):
    # Fix spacing issues: insert comma between floats if missing
    fixed_str = re.sub(r'(?<=\d)\s+(?=[+-]?\d)', ', ', embedding_str)

    try:
        return np.array(ast.literal_eval(fixed_str))
    except Exception as e:
        print(f"Error parsing:\n{embedding_str[:80]}...\n{e}")
        return None


In [ ]:
df = pd.read_csv('/content/train_with_embeddings.csv')

# Apply to both embedding columns
df['text_embedding_fixed'] = df['text_embedding'].apply(clean_embedding_string)
df['image_embedding_fixed'] = df['image_embedding'].apply(clean_embedding_string)

# Drop rows where parsing failed
df = df.dropna(subset=['text_embedding_fixed', 'image_embedding_fixed'])

# Stack into numpy arrays
X_text = np.stack(df['text_embedding_fixed'].values)
X_image = np.stack(df['image_embedding_fixed'].values)

# Target labels
y = df['label'].values


In [ ]:
df.head()

In [ ]:
X_text_flat = X_text.reshape(X_text.shape[0], -1)
X_image_flat = X_image.reshape(X_image.shape[0], -1)

# Then concatenate or use individually
X = np.concatenate([X_text_flat, X_image_flat], axis=1)  # Option C


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_text_flat = X_text.reshape(X_text.shape[0], -1)
X_image_flat = X_image.reshape(X_image.shape[0], -1)

# Then concatenate or use individually
X = np.concatenate([X_text_flat, X_image_flat], axis=1)  # Option C

# Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)

# Initialize model
input_dim = X.shape[1]
model = MLP(input_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
for epoch in range(10):  # Adjust epochs as needed
    model.train()
    for xb, yb in train_loader:
        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).argmax(dim=1).numpy()
    print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

# 1. Focal Loss definition
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# 2. Deeper MLP with BatchNorm
class MLP_Model2(nn.Module):
    def __init__(self, input_dim):
        super(MLP_Model2, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.model(x)

# 3. Data preparation
X_text_flat = X_text.reshape(X_text.shape[0], -1)
X_image_flat = X_image.reshape(X_image.shape[0], -1)
X = np.concatenate([X_text_flat, X_image_flat], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 4. Training setup
input_dim = X.shape[1]
model = MLP_Model2(input_dim)
criterion = FocalLoss(alpha=1, gamma=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

from sklearn.metrics import accuracy_score
# Modified training loop with metrics
for epoch in range(20):  # Adjust epochs as needed
    model.train()
    train_losses = []
    all_preds = []
    all_labels = []

    for xb, yb in train_loader:
        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels, all_preds)
    train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    with torch.no_grad():
        val_preds = model(X_test_tensor)
        val_loss = criterion(val_preds, y_test_tensor).item()
        val_acc = accuracy_score(y_test, torch.argmax(val_preds, dim=1).cpu().numpy())

    print(f"Epoch {epoch+1}: "
          f"Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, "
          f"Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

# 6. Evaluation
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).argmax(dim=1).numpy()
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 7. Save model2
torch.save(model.state_dict(), "model2.pth")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report
import numpy as np

# MLP model with L2 regularization and early stopping
class MLPWithRegularization(nn.Module):
    def __init__(self, input_dim):
        super(MLPWithRegularization, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),  # Dropout increased to help prevent overfitting
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),  # Dropout increased to help prevent overfitting
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)

# Early stopping class to monitor validation loss
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = np.inf
        self.stop_training = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop_training = True

# Reduce learning rate on plateau
def adjust_learning_rate(optimizer, epoch, lr_schedule={10: 1e-4, 20: 1e-5}):
    """ Reduce learning rate if plateau is detected """
    if epoch in lr_schedule:
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr_schedule[epoch]
    return optimizer

# Model initialization and training setup
input_dim = X.shape[1]
model3 = MLPWithRegularization(input_dim)
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights for imbalanced data
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# CrossEntropyLoss with class weights to address class imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model3.parameters(), lr=1e-3, weight_decay=1e-4)  # L2 regularization (weight decay)

# Early stopping setup
early_stopping = EarlyStopping(patience=5, delta=0.001)

# DataLoader setup
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Training loop with regularization and callbacks
for epoch in range(1, 21):  # Training for 20 epochs
    model3.train()
    train_losses = []
    all_preds = []
    all_labels = []

    for xb, yb in train_loader:
        preds = model3(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels, all_preds)
    train_loss = np.mean(train_losses)

    # Validation
    model3.eval()
    with torch.no_grad():
        val_preds = model3(X_test_tensor)
        val_loss = criterion(val_preds, y_test_tensor).item()
        val_acc = accuracy_score(y_test, torch.argmax(val_preds, dim=1).cpu().numpy())

    print(f"Epoch {epoch}: "
          f"Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, "
          f"Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

    # Adjust learning rate if needed
    optimizer = adjust_learning_rate(optimizer, epoch)

    # Early stopping
    early_stopping(val_loss)
    if early_stopping.stop_training:
        print("Early stopping triggered")
        break

# Final evaluation after training is completed
model3.eval()
with torch.no_grad():
    y_pred = model3(X_test_tensor).argmax(dim=1).numpy()
    print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
from sklearn.metrics import accuracy_score
# Modified training loop with metrics
for epoch in range(10):  # Adjust epochs as needed
    model.train()
    train_losses = []
    all_preds = []
    all_labels = []

    for xb, yb in train_loader:
        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels, all_preds)
    train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    with torch.no_grad():
        val_preds = model(X_test_tensor)
        val_loss = criterion(val_preds, y_test_tensor).item()
        val_acc = accuracy_score(y_test, torch.argmax(val_preds, dim=1).cpu().numpy())

    print(f"Epoch {epoch+1}: "
          f"Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, "
          f"Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

In [ ]:
def evaluate_model(model, X_test, y_test, criterion, device='cpu'):
    """
    Evaluate the model on the test set and calculate the classification metrics.

    Args:
        model (nn.Module): The trained model.
        X_test (Tensor): The input features for testing.
        y_test (Tensor): The true labels for the test set.
        criterion (nn.Module): The loss function used during training.
        device (str): The device ('cpu' or 'cuda') to evaluate the model on.

    Returns:
        dict: A dictionary containing various evaluation metrics.
    """
    model.eval()  # Set model to evaluation mode

    # Ensure tensors are on the same device as the model
    X_test, y_test = X_test.to(device), y_test.to(device)

    # Make predictions
    with torch.no_grad():
        outputs = model(X_test)
        loss = criterion(outputs, y_test).item()
        preds = torch.argmax(outputs, dim=1)

    # Calculate metrics
    accuracy = accuracy_score(y_test.cpu(), preds.cpu())
    precision = precision_score(y_test.cpu(), preds.cpu(), average='weighted')
    recall = recall_score(y_test.cpu(), preds.cpu(), average='weighted')
    f1 = f1_score(y_test.cpu(), preds.cpu(), average='weighted')

    # Classification report
    class_report = classification_report(y_test.cpu(), preds.cpu(), output_dict=True)
    confusion_mat = confusion_matrix(y_test.cpu(), preds.cpu())

    # Print the classification report and confusion matrix
    print("\nClassification Report:\n", classification_report(y_test.cpu(), preds.cpu()))
    print("Confusion Matrix:\n", confusion_mat)

    # Return metrics in a dictionary
    return {
        'loss': loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'classification_report': class_report,
        'confusion_matrix': confusion_mat
    }

# Ensure the model is moved to the correct device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model3.to(device)  # Move model to device

# Move the class weights to the same device as the model
class_weights_tensor = class_weights_tensor.to(device)

# CrossEntropyLoss with class weights to address class imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Evaluate the model after training
metrics = evaluate_model(model3, X_test_tensor, y_test_tensor, criterion, device)

# Print the metrics
print(f"\nFinal Evaluation Metrics for Model 3:")
print(f"Loss: {metrics['loss']:.4f}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import numpy as np

# Focal Loss implementation
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Apply softmax to the inputs to get probabilities
        inputs = torch.nn.functional.softmax(inputs, dim=-1)

        # Gather the probabilities corresponding to the true classes
        targets = torch.nn.functional.one_hot(targets, num_classes=inputs.size(1))
        targets = targets.float()

        # Calculate p_t
        p_t = torch.sum(inputs * targets, dim=-1)

        # Compute the focal loss
        loss = -self.alpha * (1 - p_t) ** self.gamma * torch.log(p_t + 1e-8)

        # Reduce the loss based on the specified reduction mode
        if self.reduction == 'mean':
            return torch.mean(loss)
        elif self.reduction == 'sum':
            return torch.sum(loss)
        else:
            return loss

# Final MLP Model with Focal Loss
class MLPFinalFocal(nn.Module):
    def __init__(self, input_dim):
        super(MLPFinalFocal, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),  # Increased complexity with larger first layer
            nn.ReLU(),
            nn.Dropout(0.5),  # Regularization through dropout
            nn.Linear(512, 256),  # Second hidden layer
            nn.ReLU(),
            nn.Dropout(0.5),  # Dropout to prevent overfitting
            nn.Linear(256, 128),  # Third hidden layer for deeper architecture
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),  # Fourth hidden layer for further depth
            nn.ReLU(),
            nn.Linear(64, 2)  # Output layer for binary classification
        )

    def forward(self, x):
        return self.net(x)

# Adjust learning rate on plateau
def adjust_learning_rate(optimizer, epoch, lr_schedule={10: 1e-4, 20: 1e-5}):
    if epoch in lr_schedule:
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr_schedule[epoch]
    return optimizer

# Function to evaluate the model
def evaluate_model(model, X_test, y_test, criterion, device):
    model.eval()
    with torch.no_grad():
        X_test = X_test.to(device)
        y_test = y_test.to(device)

        outputs = model(X_test)
        loss = criterion(outputs, y_test)

        _, preds = torch.max(outputs, 1)
        correct = (preds == y_test).sum().item()
        accuracy = correct / y_test.size(0)

        # Collect additional metrics
        all_labels = y_test.cpu().numpy()
        all_preds = preds.cpu().numpy()

        from sklearn.metrics import classification_report
        report = classification_report(all_labels, all_preds, output_dict=True)

        return {
            'loss': loss.item(),
            'accuracy': accuracy,
            'precision': report['accuracy'],  # Macro-average precision
            'recall': report['macro avg']['recall'],
            'f1_score': report['macro avg']['f1-score'],
        }

# Final training procedure
input_dim = X_train.shape[1]
model4 = MLPFinalFocal(input_dim).to(device)

# Define Focal Loss with class weighting
alpha = 0.25  # weight for the minority class
gamma = 2.0  # focusing parameter
criterion = FocalLoss(alpha=alpha, gamma=gamma)

# Optimizer
optimizer = optim.Adam(model4.parameters(), lr=1e-3, weight_decay=1e-4)  # L2 regularization (weight decay)

# Early stopping setup
early_stopping = EarlyStopping(patience=5, delta=0.001)

# DataLoader setup
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Training loop
for epoch in range(1, 21):  # Training for 20 epochs
    model4.train()
    train_losses = []
    all_preds = []
    all_labels = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        preds = model4(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels, all_preds)
    train_loss = np.mean(train_losses)

    # Validation
    model4.eval()
    with torch.no_grad():
        val_preds = model4(X_test_tensor)
        val_loss = criterion(val_preds, y_test_tensor).item()
        val_acc = accuracy_score(y_test, torch.argmax(val_preds, dim=1).cpu().numpy())

    print(f"Epoch {epoch}: "
          f"Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, "
          f"Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

    # Adjust learning rate if needed
    optimizer = adjust_learning_rate(optimizer, epoch)

    # Early stopping
    early_stopping(val_loss)
    if early_stopping.stop_training:
        print("Early stopping triggered")
        break

# Final evaluation after training is completed
metrics = evaluate_model(model4, X_test_tensor, y_test_tensor, criterion, device)

# Print final metrics
print("\nFinal Evaluation Metrics for Model 4 with Focal Loss:")
print(f"Loss: {metrics['loss']:.4f}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")


In [ ]:
torch.save(model.state_dict(), "model1.pth")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch

# Define the evaluation function
def evaluate_model(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_preds = []

    with torch.no_grad():  # Disable gradient calculation for evaluation
        for inputs_batch, labels_batch in dataloader:
            # Move data to the same device as model
            inputs_batch, labels_batch = inputs_batch.to(device), labels_batch.to(device)

            # Forward pass
            outputs = model(inputs_batch)

            # For binary classification, apply sigmoid and threshold at 0.5
            preds = (outputs.squeeze() > 0.5).float()

            all_labels.append(labels_batch.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    # Convert all collected labels and predictions into numpy arrays
    all_labels = np.concatenate(all_labels, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)

    # Compute evaluation metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    # AUC is applicable for binary classification, using sigmoid outputs for probability-based AUC
    auc = roc_auc_score(all_labels, all_preds) if all_preds.ndim == 1 else None

    # Print the evaluation metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    if auc is not None:
        print(f"AUC: {auc:.4f}")

    return accuracy, precision, recall, f1, auc


# Example usage of the evaluation function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Assuming you have a DataLoader for your test dataset:
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# You can call this function for each of your 5 models (e.g., MLP, CNN, etc.)

models = [model_1, model_2, model_3, model_4, model_5]  # Replace with your actual model objects

for idx, model in enumerate(models):
    print(f"Evaluating Model {idx+1}")
    model.to(device)  # Ensure the model is on the right device
    accuracy, precision, recall, f1, auc = evaluate_model(model, test_loader, device)
    print("\n" + "-"*50 + "\n")
